# ipytree Stress Test\nComprehensive widget verification after JupyterLab 4 + hatchling migration

In [ ]:
from ipytree import Tree, Node
import ipytree
print(f"ipytree version: {ipytree.__version__}")

## 1. Large tree (210 nodes, 3 levels deep)

In [ ]:
big_tree = Tree(multiple_selection=True)

for i in range(10):
    parent = Node(f"Department-{i}", icon="building", icon_style="info")
    for j in range(5):
        child = Node(f"Team-{i}-{j}", icon="users", icon_style="success")
        for k in range(3):
            leaf = Node(f"Member-{i}-{j}-{k}", icon="user", icon_style="default")
            child.add_node(leaf)
        parent.add_node(child)
    big_tree.add_node(parent)

print(f"Top-level nodes: {len(big_tree.nodes)}")
print(f"Total nodes: {sum(1 + len(c.nodes) + sum(len(g.nodes) for g in c.nodes) for c in big_tree.nodes)}")
big_tree

## 2. All icon style combinations

In [ ]:
styles = ["default", "warning", "danger", "success", "info"]
icons = ["folder", "file", "archive", "star", "heart", "cog", "home", "globe"]

style_tree = Tree()
for style in styles:
    style_node = Node(f"Style: {style}", icon="tag", icon_style=style,
                      open_icon="chevron-right", open_icon_style=style,
                      close_icon="chevron-down", close_icon_style=style)
    for icon in icons:
        style_node.add_node(Node(f"icon={icon}", icon=icon, icon_style=style))
    style_tree.add_node(style_node)

style_tree

## 3. Dynamic operations (add/remove/rename/toggle)

In [ ]:
import ipywidgets as widgets

dynamic_tree = Tree()
root = Node("Root", icon="folder", icon_style="warning",
            open_icon="angle-right", close_icon="angle-down")
dynamic_tree.add_node(root)

counter = [0]

def on_add(b):
    counter[0] += 1
    root.add_node(Node(f"Dynamic-{counter[0]}", icon="bolt", icon_style="info"))

def on_remove(b):
    if root.nodes:
        root.remove_node(root.nodes[-1])

def on_rename(b):
    if root.nodes:
        root.nodes[-1].name = f"Renamed-{counter[0]}"

def on_toggle(b):
    root.opened = not root.opened

btn_add = widgets.Button(description="Add Node", button_style="success")
btn_remove = widgets.Button(description="Remove Last", button_style="danger")
btn_rename = widgets.Button(description="Rename Last", button_style="warning")
btn_toggle = widgets.Button(description="Toggle Open/Close", button_style="info")

btn_add.on_click(on_add)
btn_remove.on_click(on_remove)
btn_rename.on_click(on_rename)
btn_toggle.on_click(on_toggle)

widgets.VBox([
    widgets.HBox([btn_add, btn_remove, btn_rename, btn_toggle]),
    dynamic_tree
])

## 4. Selection events + selected_nodes tracking

In [ ]:
select_tree = Tree(multiple_selection=True)
output = widgets.Output()

for i in range(5):
    n = Node(f"Selectable-{i}", icon="check-square", icon_style="success")
    select_tree.add_node(n)

def on_selection_change(change):
    with output:
        output.clear_output()
        names = [n.name for n in select_tree.selected_nodes]
        print(f"Selected ({len(names)}): {names}")

select_tree.observe(on_selection_change, 'selected_nodes')

widgets.VBox([select_tree, output])

## 5. Deep tree (10 levels nested)

In [ ]:
deep_tree = Tree()
current = Node("Level-0", icon="folder", opened=True)
deep_tree.add_node(current)

for depth in range(1, 11):
    child = Node(f"Level-{depth}", icon="folder" if depth < 10 else "file",
                 icon_style=["default", "info", "success", "warning", "danger"][depth % 5],
                 opened=True)
    current.add_node(child)
    current = child

deep_tree

## 6. Toggle disabled / show_icon / icon_image

In [ ]:
prop_tree = Tree()
node_a = Node("Disabled node", disabled=True, icon="lock", icon_style="danger")
node_b = Node("No icon", show_icon=False)
node_c = Node("With image icon",
              icon_image="https://upload.wikimedia.org/wikipedia/commons/thumb/3/38/Jupyter_logo.svg/44px-Jupyter_logo.svg.png")
node_d = Node("Normal node", icon="leaf", icon_style="success")

for n in [node_a, node_b, node_c, node_d]:
    prop_tree.add_node(n)

toggle_disabled = widgets.ToggleButton(description="Toggle disabled (A)")
toggle_icon = widgets.ToggleButton(description="Toggle show_icon (B)")

def on_disabled(change):
    node_a.disabled = change['new']

def on_icon(change):
    node_b.show_icon = change['new']

toggle_disabled.observe(on_disabled, 'value')
toggle_icon.observe(on_icon, 'value')

widgets.VBox([widgets.HBox([toggle_disabled, toggle_icon]), prop_tree])

## 7. Positional insert (add_node with position)

In [ ]:
pos_tree = Tree()
pos_tree.add_node(Node("First (added 1st)", icon="play", icon_style="success"))
pos_tree.add_node(Node("Third (added 2nd)", icon="play", icon_style="success"))
pos_tree.add_node(Node("Second (inserted at pos=1)", icon="exclamation-triangle", icon_style="warning"), position=1)
pos_tree.add_node(Node("Zeroth (inserted at pos=0)", icon="star", icon_style="danger"), position=0)

print("Expected order: Zeroth, First, Second, Third")
print("Actual order:", [n.name for n in pos_tree.nodes])
pos_tree

## 8. Single selection mode

In [ ]:
single_tree = Tree(multiple_selection=False)
output2 = widgets.Output()

for i in range(5):
    single_tree.add_node(Node(f"Single-{i}", icon="hand-pointer-o"))

def on_single_select(change):
    with output2:
        output2.clear_output()
        names = [n.name for n in single_tree.selected_nodes]
        print(f"Selected: {names} (should be max 1)")

single_tree.observe(on_single_select, 'selected_nodes')

print("Click multiple nodes - only one should be selected at a time")
widgets.VBox([single_tree, output2])

## 9. Scale test (1,100 / 3,300 / 5,500 nodes)

In [ ]:
import time

# Scale test: 1,000 / 3,000 / 5,000 nodes
# Each tree: N departments x 10 teams x 10 members

for dept_count, label in [(10, "1,100"), (30, "3,300"), (50, "5,500")]:
    t0 = time.time()
    tree = Tree(multiple_selection=False, animation=0)

    for i in range(dept_count):
        dept = Node(f"Dept-{i:03d}", icon="building", icon_style="info", opened=False)
        for j in range(10):
            team = Node(f"Team-{i:03d}-{j}", icon="users", icon_style="success", opened=False)
            for k in range(10):
                team.add_node(Node(f"M-{i:03d}-{j}-{k}", icon="user"))
            dept.add_node(team)
        tree.add_node(dept)

    t1 = time.time()
    print(f"--- {label} nodes: created in {t1 - t0:.2f}s ---")
    display(tree)
    print()